In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

c:\Users\User\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [9]:
notebook_folder = Path.cwd()
project_folder = notebook_folder.parent

scb_folder = (
    project_folder
    / "Year_End_2026"
    / "SCB_Account"
)

output_folder = project_folder / "Datamart"
output_folder.mkdir(parents=True, exist_ok=True)

supported_extensions = {".xlsx", ".xls"}

scb_files = sorted(
    file
    for file in scb_folder.rglob("*")
    if file.is_file()
    and file.suffix.lower() in supported_extensions
    and not file.name.startswith("~$")
)

print("SCB folder:", scb_folder)
print("Folder exists:", scb_folder.exists())
print("Total SCB files:", len(scb_files))

for file in scb_files:
    print(file.relative_to(scb_folder))

SCB folder: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Year_End_2026\SCB_Account
Folder exists: True
Total SCB files: 86
Feb\HISTSTMT_RPT2603191101367273_260319110147101.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147102.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147103.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147104.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147105.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147106.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147107.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147108.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147109.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147110.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147111.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147112.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147113.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147114.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147115.XLSX
Feb\HISTSTMT_RPT2603191101367273_260319110147116.XLSX

In [10]:
rename_map = {
    "Account Number": "account_number",
    "Account Name": "account_name",
    "Account Type": "account_type",
    "Currency Code": "currency_code",
    "Branch Code": "branch_code",
    "Date": "transaction_date",
    "Time": "transaction_time",
    "Tr Code": "transaction_code",
    "Tr Description": "transaction_type",
    "Channel": "channel",
    "Cheque No.": "cheque_number",
    "Withdrawal": "withdrawal",
    "Deposit": "deposit",
    "Outstanding Balance": "outstanding_balance",
    "Description": "description",
}

In [11]:
def read_scb_statement(file_path, root_folder):
    suffix = file_path.suffix.lower()

    if suffix == ".xlsx":
        engine = "openpyxl"
    elif suffix == ".xls":
        engine = "xlrd"
    else:
        raise ValueError(
            f"Unsupported file format: {file_path.suffix}"
        )

    df = pd.read_excel(
        file_path,
        engine=engine,
        header=0,
        dtype=str
    )

    # Clean source column names
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    df = df.rename(columns=rename_map)

    # Remove completely blank rows
    df = df.replace(
        ["", "nan", "NaN", "None"],
        np.nan
    )
    df = df.dropna(how="all").copy()

    # Source metadata
    relative_path = file_path.relative_to(root_folder)

    month_folder = (
        relative_path.parts[0]
        if len(relative_path.parts) > 1
        else None
    )

    df.insert(0, "bank", "SCB")
    df.insert(1, "month_folder", month_folder)
    df.insert(2, "source_file", file_path.name)
    df.insert(3, "source_path", str(relative_path))

    return df

In [12]:
all_scb_data = []
failed_files = []

for file in scb_files:
    try:
        file_df = read_scb_statement(
            file_path=file,
            root_folder=scb_folder
        )

        all_scb_data.append(file_df)

        account = (
            file_df["account_number"].dropna().iloc[0]
            if "account_number" in file_df.columns
            and file_df["account_number"].notna().any()
            else None
        )

        print(
            f"OK | {file.relative_to(scb_folder)} "
            f"| Account: {account} "
            f"| Rows: {len(file_df):,}"
        )

    except Exception as error:
        failed_files.append({
            "source_file": str(
                file.relative_to(scb_folder)
            ),
            "error": str(error)
        })

        print(
            f"FAILED | {file.relative_to(scb_folder)} "
            f"| {error}"
        )

c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no d

OK | Feb\HISTSTMT_RPT2603191101367273_260319110147101.XLSX | Account: 0172762156 | Rows: 100
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147102.XLSX | Account: 1022733936 | Rows: 103
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147103.XLSX | Account: 1362714171 | Rows: 100
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147104.XLSX | Account: 1564296424 | Rows: 96
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147105.XLSX | Account: 1922438369 | Rows: 99
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147106.XLSX | Account: 3164155931 | Rows: 125
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147107.XLSX | Account: 3702819949 | Rows: 122


c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no d

OK | Feb\HISTSTMT_RPT2603191101367273_260319110147108.XLSX | Account: 4122310173 | Rows: 72
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147109.XLSX | Account: 4142174890 | Rows: 101
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147110.XLSX | Account: 4221594268 | Rows: 118
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147111.XLSX | Account: 4271897496 | Rows: 67
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147112.XLSX | Account: 4282216316 | Rows: 79
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147113.XLSX | Account: 5194245562 | Rows: 85
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147114.XLSX | Account: 5692866977 | Rows: 2
OK | Feb\HISTSTMT_RPT2603191101367273_260319110147115.XLSX | Account: 6402700634 | Rows: 108


c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no d

...

c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no d

OK | May\316-เดอะโฟร์ท.XLSX | Account: 3164155931 | Rows: 164
OK | May\370-รังสิต.XLSX | Account: 3702819949 | Rows: 168
OK | May\412-วังหิน.XLSX | Account: 4122310173 | Rows: 108
OK | May\414-ขอนแก่น.XLSX | Account: 4142174890 | Rows: 131
OK | May\422-บางนา.XLSX | Account: 4221594268 | Rows: 180
OK | May\427-บางใหญ่.XLSX | Account: 4271897496 | Rows: 128
OK | May\428-คลอง3.XLSX | Account: 4282216316 | Rows: 120


c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no d

OK | May\519-นครปฐม.XLSX | Account: 5194245562 | Rows: 97
OK | May\523-ระยอง.XLSX | Account: 5234735003 | Rows: 256
OK | May\569-พิษณุโลก.XLSX | Account: 5692866977 | Rows: 111
OK | May\640-บางแสน.XLSX | Account: 6402700634 | Rows: 165
OK | May\666-โคราช.XLSX | Account: 6664038417 | Rows: 123
OK | May\669-พัทยา.XLSX | Account: 6692694546 | Rows: 106


c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no d

In [13]:
if all_scb_data:
    scb_df = pd.concat(
        all_scb_data,
        ignore_index=True,
        sort=False
    )
else:
    scb_df = pd.DataFrame()

print("Total consolidated rows:", len(scb_df))
print("Total columns:", len(scb_df.columns))

display(scb_df.head(30))

Total consolidated rows: 10747
Total columns: 20


,bank,month_folder,source_file,source_path,account_number,account_name,account_type,currency_code,branch_code,transaction_date,transaction_time,transaction_code,transaction_type,channel,cheque_number,withdrawal,deposit,outstanding_balance,description,Note
0,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,01/02/2026,03:20,X1,ฝากถอนเงินโอนไม่ใช้สมุด,ATS,NaN,NaN,"70,209.50","797,901.45",CREDIT CARD DIVISION(EDC),NaN
1,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,01/02/2026,22:59,X1,ฝากถอนเงินโอนไม่ใช้สมุด,BPAY,NaN,NaN,"147,255.00","945,156.45",รับชำระค่าสินค้าและบริการ CrossBank,NaN
2,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,02/02/2026,03:00,X1,ฝากถอนเงินโอนไม่ใช้สมุด,ATS,NaN,NaN,"20,563.02","965,719.47",CREDIT CARD DIVISION(EDC),NaN
3,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,02/02/2026,11:24,X1,ฝากถอนเงินโอนไม่ใช้สมุด,ENET,NaN,NaN,"25,035.00","990,754.47",รับโอนจาก SCB x0823 นาย ธัชกร ชยาศิสรัตน,NaN
4,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,02/02/2026,11:25,X1,ฝากถอนเงินโอนไม่ใช้สมุด,ENET,NaN,NaN,"25,035.00","1,015,789.47",รับโอนจาก SCB x0823 นาย ธัชกร ชยาศิสรัตน,NaN
5,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,02/02/2026,22:59,X1,ฝากถอนเงินโอนไม่ใช้สมุด,BPAY,NaN,NaN,"123,931.00","1,139,720.47",รับชำระค่าสินค้าและบริการ CrossBank,NaN
6,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,03/02/2026,11:54,X1,ฝากถอนเงินโอนไม่ใช้สมุด,ENET,NaN,NaN,"50,110.00","1,189,830.47",รับโอนจาก KBANK x9504 บจก. เอ็มเอ็ม ดีไซ,NaN
7,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,03/02/2026,15:49,X1,ฝากถอนเงินโอนไม่ใช้สมุด,ENET,NaN,NaN,"3,990.00","1,193,820.47",รับโอนจาก SCB x2597 นาย จักรพันธุ์ หิรัญ,NaN
8,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,03/02/2026,17:51,X1,ฝากถอนเงินโอนไม่ใช้สมุด,ENET,NaN,NaN,"15,170.00","1,208,990.47",รับโอนจาก KBANK x9463 บจก. ส่งการช่าง(20,NaN
9,SCB,Feb,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,03/02/2026,20:06,X1,ฝากถอนเงินโอนไม่ใช้สมุด,ENET,NaN,NaN,"3,240.00","1,212,230.47",รับโอนจาก KBANK x6472 น.ส. กรรณา เมืองเท,NaN


In [14]:
if "account_number" in scb_df.columns:
    scb_df["account_number"] = (
        scb_df["account_number"]
        .astype("string")
        .str.strip()
    )

In [15]:
if "transaction_date" in scb_df.columns:
    scb_df["transaction_date"] = pd.to_datetime(
        scb_df["transaction_date"],
        dayfirst=True,
        errors="coerce"
    )

amount_columns = [
    "withdrawal",
    "deposit",
    "outstanding_balance"
]

for column in amount_columns:
    if column in scb_df.columns:
        scb_df[column] = (
            scb_df[column]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({
                "": pd.NA,
                "nan": pd.NA,
                "NaN": pd.NA,
                "None": pd.NA,
                "-": pd.NA
            })
        )

        scb_df[column] = pd.to_numeric(
            scb_df[column],
            errors="coerce"
        ).astype("Float64")

In [16]:
string_columns = [
    "bank",
    "month_folder",
    "source_file",
    "source_path",
    "account_number",
    "account_name",
    "account_type",
    "currency_code",
    "branch_code",
    "transaction_time",
    "transaction_code",
    "transaction_type",
    "channel",
    "cheque_number",
    "description"
]

for column in string_columns:
    if column in scb_df.columns:
        scb_df[column] = (
            scb_df[column]
            .astype("string")
            .str.strip()
        )

In [17]:
preferred_columns = [
    "bank",
    "account_number",
    "account_name",
    "account_type",
    "currency_code",
    "branch_code",
    "month_folder",
    "transaction_date",
    "transaction_time",
    "transaction_code",
    "transaction_type",
    "withdrawal",
    "deposit",
    "outstanding_balance",
    "channel",
    "cheque_number",
    "description",
    "source_file",
    "source_path"
]

existing_columns = [
    column
    for column in preferred_columns
    if column in scb_df.columns
]

other_columns = [
    column
    for column in scb_df.columns
    if column not in existing_columns
]

scb_df = scb_df[
    existing_columns + other_columns
].copy()

display(scb_df.head(30))

,bank,account_number,account_name,account_type,currency_code,branch_code,month_folder,transaction_date,transaction_time,transaction_code,transaction_type,withdrawal,deposit,outstanding_balance,channel,cheque_number,description,source_file,source_path,Note
0,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-01,03:20,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,70209.5,797901.45,ATS,<NA>,CREDIT CARD DIVISION(EDC),HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
1,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-01,22:59,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,147255.0,945156.45,BPAY,<NA>,รับชำระค่าสินค้าและบริการ CrossBank,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
2,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-02,03:00,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,20563.02,965719.47,ATS,<NA>,CREDIT CARD DIVISION(EDC),HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
3,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-02,11:24,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,25035.0,990754.47,ENET,<NA>,รับโอนจาก SCB x0823 นาย ธัชกร ชยาศิสรัตน,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
4,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-02,11:25,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,25035.0,1015789.47,ENET,<NA>,รับโอนจาก SCB x0823 นาย ธัชกร ชยาศิสรัตน,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
5,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-02,22:59,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,123931.0,1139720.47,BPAY,<NA>,รับชำระค่าสินค้าและบริการ CrossBank,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
6,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-03,11:54,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,50110.0,1189830.47,ENET,<NA>,รับโอนจาก KBANK x9504 บจก. เอ็มเอ็ม ดีไซ,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
7,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-03,15:49,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,3990.0,1193820.47,ENET,<NA>,รับโอนจาก SCB x2597 นาย จักรพันธุ์ หิรัญ,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
8,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-03,17:51,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,15170.0,1208990.47,ENET,<NA>,รับโอนจาก KBANK x9463 บจก. ส่งการช่าง(20,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN
9,SCB,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,ออมทรัพย์,THB,0017,Feb,2026-02-03,20:06,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,3240.0,1212230.47,ENET,<NA>,รับโอนจาก KBANK x6472 น.ส. กรรณา เมืองเท,HISTSTMT_RPT2603191101367273_260319110147101.XLSX,Feb\HISTSTMT_RPT2603191101367273_2603191101471...,NaN


In [18]:
dtype_check = pd.DataFrame({
    "column_name": scb_df.columns,
    "dtype": scb_df.dtypes.astype(str).values,
    "non_null_rows": scb_df.notna().sum().values,
    "null_rows": scb_df.isna().sum().values,
    "null_percent": (
        scb_df.isna()
        .mean()
        .mul(100)
        .round(2)
        .values
    )
})

display(dtype_check)

,column_name,dtype,non_null_rows,null_rows,null_percent
0,bank,string,10747,0,0.00
1,account_number,string,10747,0,0.00
2,account_name,string,10747,0,0.00
3,account_type,string,10747,0,0.00
4,currency_code,string,10747,0,0.00
5,branch_code,string,10747,0,0.00
6,month_folder,string,10747,0,0.00
7,transaction_date,datetime64[us],10747,0,0.00
8,transaction_time,string,10747,0,0.00
9,transaction_code,string,10747,0,0.00


In [20]:
# เก็บเฉพาะรายการที่มี withdrawal หรือ deposit
valid_transactions = scb_df[
    scb_df["account_number"].notna()
    & scb_df["month_folder"].notna()
    & (
        scb_df["withdrawal"].notna()
        | scb_df["deposit"].notna()
    )
].copy()

# เรียงก่อน เพื่อเลือก transaction แรกของแต่ละบัญชีในแต่ละเดือน
valid_transactions = valid_transactions.sort_values(
    [
        "month_folder",
        "account_number",
        "transaction_date",
        "transaction_time",
        "source_file"
    ],
    na_position="last"
)

# เลือก 1 transaction ต่อ account_number ต่อเดือน
monthly_account_check = (
    valid_transactions
    .groupby(
        ["month_folder", "account_number"],
        as_index=False,
        dropna=False
    )
    .first()
)

In [21]:
check_columns = [
    "bank",
    "month_folder",
    "account_number",
    "account_name",
    "transaction_date",
    "transaction_time",
    "transaction_code",
    "transaction_type",
    "withdrawal",
    "deposit",
    "outstanding_balance",
    "channel",
    "description",
    "source_file"
]

display(
    monthly_account_check[
        [
            column
            for column in check_columns
            if column in monthly_account_check.columns
        ]
    ]
)

,bank,month_folder,account_number,account_name,transaction_date,transaction_time,transaction_code,transaction_type,withdrawal,deposit,outstanding_balance,channel,description,source_file
0,SCB,Feb,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,03:20,X1,ฝากถอนเงินโอนไม่ใช้สมุด,1200000.0,70209.5,797901.45,ATS,CREDIT CARD DIVISION(EDC),HISTSTMT_RPT2603191101367273_260319110147101.XLSX
1,SCB,Feb,1022733936,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,22:59,X1,ฝากถอนเงินโอนไม่ใช้สมุด,1400000.0,96369.0,1354755.97,BPAY,รับชำระค่าสินค้าและบริการ CrossBank,HISTSTMT_RPT2603191101367273_260319110147102.XLSX
2,SCB,Feb,1362714171,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,11:39,X1,ฝากถอนเงินโอนไม่ใช้สมุด,1500000.0,1050.0,1288043.42,ENET,รับโอนจาก BAAC x3389 นางสาว รัชดาภรณ์ สั,HISTSTMT_RPT2603191101367273_260319110147103.XLSX
3,SCB,Feb,1564296424,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,03:20,X1,ฝากถอนเงินโอนไม่ใช้สมุด,1600000.0,29716.76,1395131.06,ATS,CREDIT CARD DIVISION(EDC),HISTSTMT_RPT2603191101367273_260319110147104.XLSX
4,SCB,Feb,1922438369,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,12:10,X1,ฝากถอนเงินโอนไม่ใช้สมุด,1400000.0,10000.0,1353150.92,ENET,รับโอนจาก KBANK x5714 น.ส. อรุณโรจน์ อาช,HISTSTMT_RPT2603191101367273_260319110147105.XLSX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,SCB,May,5234735003,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-05-09,11:23,X1,ฝากถอนเงินโอนไม่ใช้สมุด,910000.0,1090.0,4090.0,ENET,รับโอนจาก TTB x0362 MR CHATTAMAT NALEK,523-ระยอง.XLSX
82,SCB,May,5692866977,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-05-01,03:34,X1,ฝากถอนเงินโอนไม่ใช้สมุด,1000000.0,4978.78,382431.58,ATS,CREDIT CARD DIVISION(EDC),569-พิษณุโลก.XLSX
83,SCB,May,6402700634,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-04-30,23:32,X1,ฝากถอนเงินโอนไม่ใช้สมุด,900000.0,3000.0,367017.87,ENET,รับโอนจาก KBANK x7507 นาย วิวัฒน์ ทอดขุน,640-บางแสน.XLSX
84,SCB,May,6664038417,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-05-01,12:02,C1,"ฝาก, ถอนเงินสด-ไม่ใช้สมุด",910000.0,43770.0,905419.29,TELL,ตลาดเซฟวัน (นครราชสีมา),666-โคราช.XLSX


In [22]:
monthly_account_check = (
    valid_transactions
    .drop_duplicates(
        subset=["month_folder", "account_number"],
        keep="first"
    )
    .reset_index(drop=True)
)

display(
    monthly_account_check[
        [
            column
            for column in check_columns
            if column in monthly_account_check.columns
        ]
    ]
)

,bank,month_folder,account_number,account_name,transaction_date,transaction_time,transaction_code,transaction_type,withdrawal,deposit,outstanding_balance,channel,description,source_file
0,SCB,Feb,0172762156,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,03:20,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,70209.5,797901.45,ATS,CREDIT CARD DIVISION(EDC),HISTSTMT_RPT2603191101367273_260319110147101.XLSX
1,SCB,Feb,1022733936,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,22:59,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,96369.0,1354755.97,BPAY,รับชำระค่าสินค้าและบริการ CrossBank,HISTSTMT_RPT2603191101367273_260319110147102.XLSX
2,SCB,Feb,1362714171,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,11:39,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,1050.0,1288043.42,ENET,รับโอนจาก BAAC x3389 นางสาว รัชดาภรณ์ สั,HISTSTMT_RPT2603191101367273_260319110147103.XLSX
3,SCB,Feb,1564296424,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,03:20,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,29716.76,1395131.06,ATS,CREDIT CARD DIVISION(EDC),HISTSTMT_RPT2603191101367273_260319110147104.XLSX
4,SCB,Feb,1922438369,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-02-01,12:10,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,10000.0,1353150.92,ENET,รับโอนจาก KBANK x5714 น.ส. อรุณโรจน์ อาช,HISTSTMT_RPT2603191101367273_260319110147105.XLSX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,SCB,May,5234735003,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-05-09,11:23,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,1090.0,4090.0,ENET,รับโอนจาก TTB x0362 MR CHATTAMAT NALEK,523-ระยอง.XLSX
82,SCB,May,5692866977,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-05-01,03:34,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,4978.78,382431.58,ATS,CREDIT CARD DIVISION(EDC),569-พิษณุโลก.XLSX
83,SCB,May,6402700634,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-04-30,23:32,X1,ฝากถอนเงินโอนไม่ใช้สมุด,<NA>,3000.0,367017.87,ENET,รับโอนจาก KBANK x7507 นาย วิวัฒน์ ทอดขุน,640-บางแสน.XLSX
84,SCB,May,6664038417,บริษัท ไอ แฮฟ ซีพียู จำกัด,2026-05-01,12:02,C1,"ฝาก, ถอนเงินสด-ไม่ใช้สมุด",<NA>,43770.0,905419.29,TELL,ตลาดเซฟวัน (นครราชสีมา),666-โคราช.XLSX


In [23]:
account_count_check = (
    scb_df
    .groupby("month_folder")["account_number"]
    .nunique()
    .rename("total_accounts")
    .reset_index()
)

sample_count_check = (
    monthly_account_check
    .groupby("month_folder")["account_number"]
    .nunique()
    .rename("sampled_accounts")
    .reset_index()
)

count_validation = account_count_check.merge(
    sample_count_check,
    on="month_folder",
    how="left"
)

count_validation["check"] = np.where(
    count_validation["total_accounts"]
    == count_validation["sampled_accounts"],
    "OK",
    "MISSING ACCOUNT"
)

display(count_validation)

,month_folder,total_accounts,sampled_accounts,check
0,Feb,17,17,OK
1,Jan,16,16,OK
2,June,18,18,OK
3,Mar,17,17,OK
4,May,18,18,OK
